# PhysioLive - M2 (Squat)

**How to run:** `Kernel > Restart & Run All`. A browser tab opens at `http://localhost:8000` with the live camera, skeleton, HUD and per-rep verdict. Stop the loop with `Kernel > Interrupt` (or ctrl+c in the terminal running Jupyter).

**Requirements:** `pip install -r requirements.txt`. First run downloads `yolov8s-pose.pt` (~22 MB). Optional: run `python -c "from ultralytics import YOLO; YOLO('yolov8s-pose.pt').export(format='openvino', imgsz=384)"` once to get the OpenVINO engine for a 2-3x CPU speedup - the pose loader picks it up automatically.

**Scope of this milestone:** Squat only, front-camera view, Hebrew voice feedback. Form rules: depth (ROM), knee-over-toe, torso lean.


In [ ]:
import json
import sys
import time
import webbrowser
from pathlib import Path

import cv2

PROJECT_ROOT = Path().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from app.pose_gate import PoseInferencer, draw_skeleton, KP_MIN_CONF
from app.angles import all_angles
from app.rep_counter import RepCounter
from app.form_rules import evaluate as evaluate_rules
from app.voice import VoiceWorker
from app.dashboard_server import DashboardServer, STATE
from app.phone_stream import open_source

print(f"project root: {PROJECT_ROOT}")


## 1. Load exercise config


In [ ]:
EXERCISE_ID = "squat"
LANGUAGE = "he"   # 'he' or 'en'

exercise_path = SRC / "app" / "exercises" / f"{EXERCISE_ID}.json"
with open(exercise_path, "r", encoding="utf-8") as f:
    EXERCISE = json.load(f)
print(f"loaded exercise: {EXERCISE['display_name_en']} ({EXERCISE_ID})")
print(f"pose backend:    {EXERCISE['pose_backend']}")
print(f"rep goal:        {EXERCISE['rep_goal']}")


## 2. Warm the pose model
First call downloads / initializes the checkpoint. OpenVINO engine is picked up automatically if present next to the .pt file.


In [ ]:
pose = PoseInferencer(backend=EXERCISE["pose_backend"], imgsz=384)
import numpy as np
_ = pose.infer(np.zeros((384, 384, 3), dtype=np.uint8))
print("pose model ready.")


## 3. Start the dashboard and the voice worker


In [ ]:
server = DashboardServer(port=8000, directory=SRC / "web")
url = server.start()
voice = VoiceWorker(language=LANGUAGE)
voice.start()
print(f"dashboard: {url}")
try:
    webbrowser.open(url, new=2)
except Exception:
    pass


## 4. Open the video source
Change `SOURCE` to `1` for a second webcam, an RTSP / MJPEG URL for a phone stream (e.g. `"http://192.168.1.42:8080/video"`), or a local mp4 path.


In [ ]:
SOURCE = 0
cap = open_source(SOURCE, width=1280, height=720, fps=30)
ok, probe = cap.read()
if not ok:
    raise RuntimeError(f"cannot read from source {SOURCE!r}")
print(f"source open: frame shape {probe.shape}")


## 5. Live loop

Reads frames, runs pose, updates rep counter, evaluates rules on rep-end, speaks feedback and pushes the annotated frame to the browser as MJPEG. `Kernel > Interrupt` to stop.


In [ ]:
rep_def = EXERCISE["rep_definition"]
rep_counter = RepCounter(
    standing_deg=rep_def["standing_deg"],
    bottom_deg=rep_def["bottom_deg"],
    hysteresis_deg=rep_def["hysteresis_deg"],
    confirm_frames=rep_def["confirm_frames"],
)

STATE.set_state({
    "running": True,
    "exercise": EXERCISE["display_name_he" if LANGUAGE == "he" else "display_name_en"],
    "rep_count": 0,
    "rep_goal": EXERCISE["rep_goal"],
    "rep_state": rep_counter.state,
    "knee_angle": None,
    "verdict": {"level": "", "text": ""},
})


def _pick_primary_side(angles):
    kl = angles.get("knee_left")
    kr = angles.get("knee_right")
    if kl is not None and kr is not None:
        return "left" if kl <= kr else "right"
    if kl is not None:
        return "left"
    if kr is not None:
        return "right"
    return None


def _hud(frame, rep_count, rep_goal, knee_angle, state_text, verdict_text,
         verdict_level):
    h, w = frame.shape[:2]
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (360, 130), (16, 20, 28), -1)
    cv2.addWeighted(overlay, 0.72, frame, 0.28, 0, frame)
    cv2.putText(frame, f"Reps  {rep_count} / {rep_goal}", (24, 46),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (232, 236, 241), 2, cv2.LINE_AA)
    knee_txt = f"Knee  {int(knee_angle)}deg" if knee_angle is not None else "Knee  -"
    cv2.putText(frame, knee_txt, (24, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (154, 164, 178), 2, cv2.LINE_AA)
    cv2.putText(frame, f"State {state_text}", (24, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (154, 164, 178), 2, cv2.LINE_AA)
    if verdict_text:
        color = {"good": (71, 199, 106), "warn": (36, 165, 245),
                 "bad": (68, 68, 239)}.get(verdict_level, (232, 236, 241))
        cv2.rectangle(frame, (10, h - 60), (min(w - 10, 900), h - 10),
                      (16, 20, 28), -1)
        cv2.putText(frame, verdict_text[:70], (24, h - 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2, cv2.LINE_AA)
    return frame


last_verdict_text = ""
last_verdict_level = ""
loop_t0 = time.perf_counter()
frames_seen = 0

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        kps = pose.infer(frame)
        angles = all_angles(kps) if kps else {}
        primary_side = _pick_primary_side(angles) if angles else None
        knee_angle = angles.get(f"knee_{primary_side}") if primary_side else None
        torso_vert = angles.get("torso_vertical") if angles else None
        torso_len = angles.get("torso_length") if angles else None
        knee_toe_norm = None
        if primary_side and angles.get(f"knee_over_toe_{primary_side}") is not None and torso_len:
            knee_toe_norm = angles[f"knee_over_toe_{primary_side}"] / torso_len

        event = rep_counter.update(knee_angle, torso_vertical=torso_vert,
                                   knee_over_toe_norm=knee_toe_norm)
        if event is not None:
            verdict = evaluate_rules(event.sample, EXERCISE["rules"],
                                     language=LANGUAGE)
            last_verdict_text = (verdict.text_he if LANGUAGE == "he"
                                 else verdict.text_en)
            last_verdict_level = verdict.level
            voice.say(last_verdict_text)

        if kps:
            frame = draw_skeleton(frame, kps, min_conf=KP_MIN_CONF)
        frame = _hud(frame, rep_counter.count, EXERCISE["rep_goal"],
                     knee_angle, rep_counter.state,
                     last_verdict_text, last_verdict_level)

        STATE.set_state({
            "running": True,
            "exercise": EXERCISE["display_name_he" if LANGUAGE == "he" else "display_name_en"],
            "rep_count": rep_counter.count,
            "rep_goal": EXERCISE["rep_goal"],
            "rep_state": rep_counter.state,
            "knee_angle": knee_angle,
            "verdict": {"level": last_verdict_level,
                        "text": last_verdict_text},
        })

        ok_enc, jpeg = cv2.imencode(".jpg", frame,
                                    [int(cv2.IMWRITE_JPEG_QUALITY), 72])
        if ok_enc:
            STATE.push_frame(jpeg.tobytes())

        frames_seen += 1
        if frames_seen % 60 == 0:
            fps = frames_seen / max(1e-6, time.perf_counter() - loop_t0)
            print(f"~{fps:.1f} fps, reps: {rep_counter.count}")
except KeyboardInterrupt:
    print("interrupted by user.")
finally:
    cap.release()
    voice.stop()
    STATE.set_state({**STATE.get_state(), "running": False})
    print(f"final rep count: {rep_counter.count}")
